In [ ]:
# ==============================================================================
# JUPYTER NOTEBOOK: AUTOMATED SOLAR TRANSIT ANALYSIS WITH ZWO SEESTAR
# ==============================================================================

# %% [markdown]
# # Measuring and Adjusting Solar Transit Time with a Seestar Telescope
#
# This notebook automates the measurement of the Sun's transit time over its own diameter. 
# It uses **OpenCV** to track the center of the Sun across video frames, extracts the raw pixel drift, 
# visualizes the sensor geometry, and applies astronomical corrections for:
# 1. **Declination (\(\delta\)):** Accounting for the slower apparent movement of the Sun away from the celestial equator.
# 2. **Path Angle (\(\theta\)):** Accounting for the diagonal drift across the camera sensor grid.
#
# ---

# %% [markdown]
# ## Step 1: Dependencies and Environment Setup
# We will use `opencv-python` for video frame processing, `numpy` for geometry, and `matplotlib` for generating our sensor path visualization.

# %%
# Install opencv-python if running in a cloud environment like Google Colab
# !pip install opencv-python-headless

import cv2
import numpy as np
import matplotlib.pyplot as plt
import math
from IPython.display import HTML
from base64 import b64encode

print("Libraries successfully imported.")

# %% [markdown]
# ## Step 2: Visualizing the Sensor Geometry & Path Angle (\(\theta\))
#
# The graphic below models the **Seestar IMX462 sensor frame** (1920x1080 pixels). 
# Because the telescope is completely stationary and its sensor orientation may not perfectly match the celestial grid, 
# the Sun drifts at a **Path Angle (\(\theta\))**. 
#
# *Note: The **Declination Angle (\(\delta\))** represents how far north or south of the celestial equator the Sun is in the sky, which dictates the overall speed of the drift. The **Path Angle (\(\theta\))** is the geometric tilt of the path cutting across your physical camera pixels.*

# %%
# Setup a simulated 1920x1080 Seestar frame
fig, ax = plt.subplots(figsize=(10, 5.625))
ax.set_xlim(0, 1920)
ax.set_ylim(0, 1080)
ax.invert_yaxis() # Match standard image coordinate systems (0,0 at top-left)

# Draw Seestar Sensor boundaries
ax.set_facecolor('#1e1e24')
fig.patch.set_facecolor('#0f0f12')
ax.spines['top'].set_color('#ffffff')
ax.spines['bottom'].set_color('#ffffff')
ax.spines['left'].set_color('#ffffff')
ax.spines['right'].set_color('#ffffff')

# Define solar drift path (diagonal example)
theta_example = 15 # Path angle in degrees
x_path = np.linspace(100, 1820, 100)
# Equation of line tracking the drift path center
y_center = 750 - (x_path - 100) * math.tan(math.radians(theta_example))
ax.plot(x_path, y_center, color='#ff9f1c', linestyle='--', linewidth=2, label="Sun Center Path")

# Draw the Sun at entering and exiting points
sun_radius = 180
sun_start = plt.Circle((350, y_center[15]), sun_radius, color='#ffcf33', alpha=0.8, label="Sun Position")
sun_end = plt.Circle((1550, y_center[85]), sun_radius, color='#ffcf33', alpha=0.5)
ax.add_patch(sun_start)
ax.add_patch(sun_end)

# Draw Reference Horizontal Line for Angle Theta
ax.axhline(y=y_center[15], xmin=0.18, xmax=0.5, color='#4ea8de', linestyle=':', linewidth=2, label="Pixel Row Reference")

# Labeling text
ax.text(650, y_center[15] - 30, r"Path Angle (\(\theta\))", color='#4ea8de', fontsize=12, fontweight='bold')
ax.text(350, y_center[15] + 10, "Leading Edge (T1)", color='black', fontsize=9, ha='center', fontweight='bold')
ax.text(1550, y_center[85] + 10, "Trailing Edge (T2)", color='black', fontsize=9, ha='center', fontweight='bold')

# Display Settings
ax.set_title("Seestar Sensor Frame Geometry & Solar Drift Path", color='white', fontsize=14, pad=15)
ax.set_xlabel("Sensor Width (Pixels)", color='white')
ax.set_ylabel("Sensor Height (Pixels)", color='white')
ax.tick_params(colors='white')
ax.grid(color='#33333b', linestyle=':')
ax.legend(loc="upper left")

plt.show()

# %% [markdown]
# ## Step 3: Computer Vision Tracking via OpenCV
# This code block parses an uploaded Seestar solar video file. It filters for the bright solar disk using simple image thresholding, finds the contour, calculates its center point, and records the timestamps of the tracking run.

# %%
video_path = "seestar_solar_transit.mp4" # Replace with your actual video filename
cap = cv2.VideoCapture(video_path)

# Initialize fallback defaults in case video is not found/uploaded yet
video_found = False
centers_x = []
centers_y = []
timestamps = []
fps = 30.0

if not cap.isOpened():
    print(f"Warning: Could not open video file '{video_path}'. Sandbox mode enabled with dummy values.")
else:
    video_found = True
    fps = cap.get(cv2.CAP_PROP_FPS)
    frame_count = 0

    while True:
        ret, frame = cap.read()
        if not ret:
            break # Video ended
            
        frame_count += 1
        current_time_sec = frame_count / fps
        
        # Convert to grayscale
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        
        # Threshold the image to isolate the ultra-bright sun disk
        _, thresh = cv2.threshold(gray, 240, 255, cv2.THRESH_BINARY)
        
        # Find contours of the isolated sun
        contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        
        if contours:
            # Take the largest detected shape
            largest_contour = max(contours, key=cv2.contourArea)
            
            # Calculate moments to find the center coordinate (X, Y)
            M = cv2.moments(largest_contour)
            if M["m00"] != 0:
                cX = int(M["m10"] / M["m00"])
                cY = int(M["m01"] / M["m00"])
                
                centers_x.append(cX)
                centers_y.append(cY)
                timestamps.append(current_time_sec)
                
    cap.release()
    print(f"Successfully processed {frame_count} video frames at {fps} FPS.")

# %% [markdown]
# ## Step 4: Extract Path Angle (\(\theta\)) and Raw Transit Time
# Using linear regression on the tracked pixel coordinates, we calculate the exact path angle (\(\theta\)) across the sensor grid. 
# If no video is present, the script auto-injects preset values so you can see how the downstream math works.

# %%
if video_found and len(centers_x) > 5:
    # Fit line to find the actual Path Angle (theta)
    slope, intercept = np.polyfit(centers_x, centers_y, 1)
    drift_angle_rad = math.atan(slope)
    drift_angle_deg = abs(math.degrees(drift_angle_rad))
    
    # Extract Raw Transit Time from Frame Data
    raw_transit_time = timestamps[-1] - timestamps[0]
    print("--- REAL DATA LOGGED ---")
else:
    # Baseline simulation fallbacks for notebook testing
    drift_angle_deg = 15.0 
    raw_transit_time = 132.5
    print("--- REPLAYING PRESET ENVIRONMENT FALLBACKS ---")

print(f"Calculated Path Angle (Theta): {drift_angle_deg:.2f}°")
print(f"Raw Extracted Transit Time:    {raw_transit_time:.2f} seconds")

# %% [markdown]
# ## Step 5: Execute Ephemeris Corrections
# Enter today's exact Solar Declination angle (\(\delta\)). The script combines both the **Declination (\(cos\delta\))** correction and the **Path (\(cos\theta\))** correction to yield the true physical solar diameter transit duration.
#
# \[\text{True Transit Time} = \text{Raw Time} \times \cos(\delta) \times \cos(\theta)\]

# %%
# --- CHANGE THIS BASED ON YOUR OBSERVATION DATE ---
# Solar Declination from Stellarium (+23.5° in June, 0° in Sept/March, -23.5° in Dec)
declination_deg = 20.1 

# Process math (converting degrees to radians for Python)
declination_rad = math.radians(declination_deg)
drift_angle_rad = math.radians(drift_angle_deg)

# Combined Formula
corrected_transit_time = raw_transit_time * math.cos(declination_rad) * math.cos(drift_angle_rad)

# Earth rate translation to angular diameter (15 degrees / 3600 seconds)
rotation_rate_deg_per_sec = 15.0 / 3600.0
measured_angular_diameter = corrected_transit_time * rotation_rate_deg_per_sec

print(f"--- MATHEMATICAL ADJUSTMENT RESULTS ---")
print(f"True Equatorial Transit Time:   {corrected_transit_time:.2f} seconds")
print(f"Derived Solar Angular Diameter:  {measured_angular_diameter:.4f}°")
print(f"Actual Target Solar Diameter:    ~0.5333°")

# Calculate percent error
expected_diameter = 0.5333
percent_error = abs((measured_angular_diameter - expected_diameter) / expected_diameter) * 100
print(f"Calculation Error Rate:          {percent_error:.2f}%")

# %% [markdown]
# ## Step 6: Generate and Save an Animated Tracking Overlay Video
# This block re-opens your original clip and draws a targeting marker, a historical path line, 
# and real-time calculation data onto every frame, rendering it back out as a standalone `.mp4` file.

# %%
if not video_found:
    print("Skipping video compilation step: No raw source clip found to modify.")
else:
    cap = cv2.VideoCapture(video_path)
    frame_width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    
    # Configure the MP4 output writer
    output_path = "seestar_solar_transit_tracked.mp4"
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(output_path, fourcc, fps, (frame_width, frame_height))
    
    frame_idx = 0
    print("Encoding output visualization video...")

    while True:
        ret, frame = cap.read()
        if not ret:
            break
            
        if frame_idx < len(centers_x):
            current_x = centers_x[frame_idx]
            current_y = centers_y[frame_idx]
            
            # Draw trail mapping path movement up to current frame
            for i in range(1, frame_idx + 1):
                cv2.line(frame, (centers_x[i-1], centers_y[i-1]), (centers_x[i], centers_y[i]), (0, 159, 255), 2)
                
            # Draw targeting crosshair
            cv2.drawMarker(frame, (current_x, current_y), (0, 0, 255), markerType=cv2.MARKER_CROSS, markerSize=30, thickness=2)
            


In [ ]:
<!---
# Print telemetry canvas datacv2.putText(frame, f"Frame: {frame_idx}", (50, 80), cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 255, 255), 2)cv2.putText(frame, f"X: {current_x}px | Y: {current_y}px", (50, 130), cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 255, 255), 2)cv2.putText(frame, f"Path Angle: {drift_angle_deg:.1f} deg", (50, 180), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 159, 255), 2)out.write(frame)frame_idx += 1cap.release()out.release()print(f"Success! Processed track file exported as: '{output_path}'")%% [markdown]## Step 7: Inline Notebook Video PlaybackRun this cell to generate an HTML5 video playback wrapper to stream your newly rendered tracking data file directly within the browser interface.%%if video_found:mp4_data = open('seestar_solar_transit_tracked.mp4','rb').read()data_url = "data:video/mp4;base64," + b64encode(mp4_data).decode()HTML(f"""""")else:print("Notebook is running in simulated dataset mode. Upload a raw transit file to populate inline components.")
--->